# RepFR Analyses Runner - RepeatedRecallsKahanaJacobs2000

Run serial recall analyses for Kahana & Jacobs 2000 dataset via papermill.

**Paradigm:** Serial Recall with item repetitions
**Reference:** @kahana2000interresponse

In [ ]:
import os
import numpy as np
import papermill as pm

from jaxcmr.helpers import find_project_root, generate_trial_mask, load_data

## Dataset Configuration

In [ ]:
# ============================================================================
# DATASET CONFIGURATION - RepeatedRecallsKahanaJacobs2000
# ============================================================================
# Paradigm: Serial Recall with item repetitions
# Reference: @kahana2000interresponse
# Queries from thesis work/thesis/KahanaJacobs2000.ipynb
# ============================================================================

DATA_NAME = "RepeatedRecallsKahanaJacobs2000"
DATA_FILE = "RepeatedRecallsKahanaJacobs2000.h5"

# Trial queries from thesis (serial recall specific)
# Uses recall_attempt and recall_total to select first forward recall
TRIAL_QUERY = "jnp.logical_and(data['recall_attempt'] == 1, data['recall_total'] > 0)"
MIXED_TRIAL_QUERY = "jnp.logical_and(jnp.logical_and(data['recall_attempt'] == 1, data['recall_total'] > 0), data['repetitions'] == 1)"
CONTROL_TRIAL_QUERY = "jnp.logical_and(jnp.logical_and(data['recall_attempt'] == 1, data['recall_total'] > 0), data['repetitions'] == 0)"

# Analyses from thesis KahanaJacobs2000.ipynb
SINGLE_ANALYSIS_PATHS = [
    "jaxcmr.analyses.serialrepcrp.plot_rep_crp",
]

COMPARISON_ANALYSIS_PATHS = [
    "jaxcmr.analyses.spc.plot_spc",
    "jaxcmr.analyses.crp.plot_crp",
    "jaxcmr.analyses.pnr.plot_pnr",
    "jaxcmr.analyses.serialrepcrp.plot_first_rep_crp",
    "jaxcmr.analyses.serialrepcrp.plot_second_rep_crp",
]

In [ ]:
project_root = find_project_root()
templates_dir = os.path.join(project_root, "templates")
rendered_dir = os.path.join(project_root, "projects/repfr/notebooks/rendered")
figures_dir = os.path.join(project_root, "projects/repfr/results/figures/analyses")

os.makedirs(rendered_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)

data_path = os.path.join(project_root, "data", DATA_FILE)

if not os.path.exists(data_path):
    print(f"WARNING: Data file not found at {data_path}")
    DATA_READY = False
else:
    data = load_data(data_path)
    DATA_READY = True
    print(f"Loaded data from {data_path}")

## Analysis Specifications

Following thesis pattern from `work/thesis/KahanaJacobs2000.ipynb`:
- **Single analyses**: serialrepcrp (mixed and control separately)
- **Comparison analyses**: spc, crp, pnr, first/second rep CRP (mixed vs control)

In [ ]:
analysis_specs = [
    # Basic benchmarks
    {
        "name": "spc",
        "template": "spc.ipynb",
        "parameters": {
            "run_tag": "SPC",
            "data_name": DATA_NAME,
            "data_query": MIXED_TRIAL_QUERY,
            "output_path": os.path.join(figures_dir, f"spc_{DATA_NAME}.png"),
        },
    },
    {
        "name": "crp",
        "template": "crp.ipynb",
        "parameters": {
            "run_tag": "CRP",
            "data_name": DATA_NAME,
            "data_query": MIXED_TRIAL_QUERY,
            "output_path": os.path.join(figures_dir, f"crp_{DATA_NAME}.png"),
        },
    },
    {
        "name": "pnr",
        "template": "pnr.ipynb",
        "parameters": {
            "run_tag": "PNR",
            "data_name": DATA_NAME,
            "data_query": MIXED_TRIAL_QUERY,
            "query_recall_position": 0,
            "output_path": os.path.join(figures_dir, f"pnr_{DATA_NAME}.png"),
        },
    },
    # Serial repetition CRP
    {
        "name": "serialrepcrp",
        "template": "serialrepcrp.ipynb",
        "parameters": {
            "data_name": DATA_NAME,
            "data_query": MIXED_TRIAL_QUERY,
            "ctrl_query": CONTROL_TRIAL_QUERY,
            "output_path": os.path.join(figures_dir, f"serialrepcrp_{DATA_NAME}.png"),
        },
    },
]

## Run Analyses

In [ ]:
if not DATA_READY:
    print("Skipping analyses - data not yet available.")
else:
    for spec in analysis_specs:
        template_path = os.path.join(templates_dir, spec["template"])
        
        if not os.path.exists(template_path):
            print(f"WARNING: Template not found: {template_path}")
            continue
        
        rendered_name = f"{spec['name']}_{DATA_NAME}.ipynb"
        output_path = os.path.join(rendered_dir, rendered_name)
        
        print(f"Running {spec['name']} -> {output_path}")
        pm.execute_notebook(
            template_path,
            output_path,
            parameters=spec["parameters"],
        )